# Capítulo 3: El Modelo Maldito (Regresión Lineal y Logística)

## La Prueba de Fuego del Machine Learning

> *"Si tu modelo complejo no supera a una regresión con 3 variables, no sirve."*

En este notebook implementaremos:
1. **Regresión Lineal** para predecir ingresos totales
2. **Regresión Logística** para predecir suscripciones al programa de lealtad
3. **Comparación** y análisis de qué modelo funciona mejor
4. **Post-Mortem** para aprender de los errores

In [ ]:
# Celda 1: Importaciones

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuración de gráficos
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("Importaciones completas. El martillo está listo.")

In [ ]:
# Celda 2: Carga de datos de Coffee Shop Sales

df = pd.read_csv('../datos/datos_cafeteria.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"Dataset cargado: {df.shape[0]} días, {df.shape[1]} columnas")
print(f"\nPrimeras 5 filas:")
df.head()

In [ ]:
# Celda 3: EDA Rápido

print("=" * 60)
print("ANÁLISIS EXPLORATORIO RÁPIDO")
print("=" * 60)

# Información general
print("\n1. INFORMACIÓN GENERAL:")
print(f"   Shape: {df.shape}")
print(f"   Columnas: {list(df.columns)}")
print(f"   Valores nulos: {df.isnull().sum().sum()}")
print(f"   Período: {df['date'].min()} a {df['date'].max()}")

# Estadísticas descriptivas
print("\n2. ESTADÍSTICAS DESCRIPTIVAS:")
df.describe().round(2)

# Distribución por día de la semana
print("\n3. DISTRIBUCIÓN POR DÍA DE LA SEMANA:")
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
transacciones_por_dia = df.groupby('day_of_week')['transactions'].mean().reindex(day_order)
print(transacciones_por_dia.round(1))

# Festivos vs normales
print("\n4. FESTIVOS VS NORMALES:")
festivos = df[df['is_holiday'] == 1]
normales = df[df['is_holiday'] == 0]
print(f"   Días festivos: {len(festivos)}")
print(f"   Transacciones promedio (festivo): {festivos['transactions'].mean():.1f}")
print(f"   Transacciones promedio (normal): {normales['transactions'].mean():.1f}")
print(f"   Diferencia: {((festivos['transactions'].mean() / normales['transactions'].mean()) - 1) * 100:.1f}%")

# Correlaciones clave
print("\n5. CORRELACIONES CON total_revenue:")
corr_revenue = df[['transactions', 'avg_order_value', 'is_holiday', 'loyalty_signups']].corrwith(df['total_revenue'])
print(corr_revenue.round(3))

# Visualizaciones
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Transacciones por día
df.boxplot(column='transactions', by='day_of_week', ax=axes[0, 0])
axes[0, 0].set_title('Transacciones por Día de la Semana')
axes[0, 0].set_xlabel('Día')
axes[0, 0].set_ylabel('Transacciones')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Loyalty signups por clima
df.boxplot(column='loyalty_signups', by='weather', ax=axes[0, 1])
axes[0, 1].set_title('Suscripciones por Clima')
axes[0, 1].set_xlabel('Clima')
axes[0, 1].set_ylabel('Suscripciones')

# 3. Heatmap de correlación
corr_matrix = df[['transactions', 'avg_order_value', 'total_revenue', 'loyalty_signups', 'is_holiday']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=axes[1, 0])
axes[1, 0].set_title('Matriz de Correlación')

# 4. Serie temporal de ingresos
axes[1, 1].plot(df['date'], df['total_revenue'], alpha=0.7, linewidth=0.8)
axes[1, 1].set_title('Ingresos Totales a lo Largo del Tiempo')
axes[1, 1].set_xlabel('Fecha')
axes[1, 1].set_ylabel('Ingresos ($)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Detección de outliers
print("\n6. OUTLIERS (valores > Q3 + 1.5*IQR):")
for col in ['transactions', 'total_revenue', 'loyalty_signups']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[df[col] > Q3 + 1.5 * IQR]
    print(f"   {col}: {len(outliers)} outliers")

In [ ]:
# Celda 4: Regresión Lineal (predecir total_revenue)

print("=" * 60)
print("REGRESIÓN LINEAL: Prediciendo total_revenue")
print("=" * 60)

# Preparar datos
X = df[['transactions', 'avg_order_value', 'is_holiday']]
y = df['total_revenue']

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nDatos de entrenamiento: {X_train.shape[0]} muestras")
print(f"Datos de prueba: {X_test.shape[0]} muestras")

# Estandarizar
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Entrenar modelo
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predicciones
y_train_pred = lr_model.predict(X_train_scaled)
y_test_pred = lr_model.predict(X_test_scaled)

print("\nModelo entrenado exitosamente.")
print(f"\nEcuación del modelo:")
print(f"total_revenue = {lr_model.intercept_:.2f} + {lr_model.coef_[0]:.2f}*transactions + {lr_model.coef_[1]:.2f}*avg_order_value + {lr_model.coef_[2]:.2f}*is_holiday")

In [ ]:
# Celda 5: Interpretación de coeficientes

print("=" * 60)
print("INTERPRETACIÓN DE COEFICIENTES")
print("=" * 60)

# Crear DataFrame de coeficientes
coef_df = pd.DataFrame({
    'Variable': ['Intercepto', 'transactions', 'avg_order_value', 'is_holiday'],
    'Coeficiente': [lr_model.intercept_] + list(lr_model.coef_)
})

print("\nCoeficientes del modelo:")
print(coef_df.to_string(index=False))

print("\n" + "-" * 60)
print("INTERPRETACIÓN EN CONTEXTO DE NEGOCIO:")
print("-" * 60)

print(f"\n1. TRANSACTIONS (β₁ = {lr_model.coef_[0]:.2f}):")
print(f"   Por cada transacción adicional, los ingresos aumentan ${lr_model.coef_[0]:.2f}")
print(f"   Interpretación: Más clientes = más ingresos (obvio, pero cuantificado)")

print(f"\n2. AVG_ORDER_VALUE (β₂ = {lr_model.coef_[1]:.2f}):")
print(f"   Por cada dólar adicional en el valor promedio del pedido, los ingresos")
print(f"   aumentan ${lr_model.coef_[1]:.2f}")
print(f"   Interpretación: Pedidos más caros = más ingresos (esperado)")

print(f"\n3. IS_HOLIDAY (β₃ = {lr_model.coef_[2]:.2f}):")
print(f"   Los días festivos generan ${lr_model.coef_[2]:.2f} MÁS que los días normales")
print(f"   Interpretación: Festivos son buenos para el negocio")

print(f"\n4. INTERCEPTO (β₀ = {lr_model.intercept_:.2f}):")
print(f"   Ingresos base cuando todas las variables son 0")
print(f"   Interpretación: No tiene sentido físico, es solo matemático")

# Verificar correlación entre variables independientes
print("\n" + "-" * 60)
print("VERIFICACIÓN DE MULTICOLINEALIDAD:")
print("-" * 60)
corr_independent = X_train.corr()
print("\nCorrelación entre variables independientes:")
print(corr_independent.round(3))
print("\n¿Hay multicolinealidad problemática? ", end="")
if corr_independent.abs().values[np.triu_indices_from(corr_independent.values, k=1)].max() > 0.7:
    print("SÍ - Considerar eliminar una variable")
else:
    print("NO - Las variables son independientes")

In [ ]:
# Celda 6: Métricas R², RMSE

print("=" * 60)
print("MÉTRICAS DE REGRESIÓN LINEAL")
print("=" * 60)

# Calcular métricas
metrics = {
    'R² (Train)': r2_score(y_train, y_train_pred),
    'R² (Test)': r2_score(y_test, y_test_pred),
    'RMSE (Train)': np.sqrt(mean_squared_error(y_train, y_train_pred)),
    'RMSE (Test)': np.sqrt(mean_squared_error(y_test, y_test_pred)),
    'MAE (Train)': mean_absolute_error(y_train, y_train_pred),
    'MAE (Test)': mean_absolute_error(y_test, y_test_pred)
}

print("\nMétricas de rendimiento:")
for metric, value in metrics.items():
    print(f"   {metric}: {value:.4f}")

# Análisis de overfitting
print("\n" + "-" * 60)
print("ANÁLISIS DE OVERFITTING:")
print("-" * 60)
r2_diff = metrics['R² (Train)'] - metrics['R² (Test)']
rmse_ratio = metrics['RMSE (Test)'] / metrics['RMSE (Train)']

print(f"\nDiferencia R² (Train - Test): {r2_diff:.4f}")
print(f"Ratio RMSE (Test/Train): {rmse_ratio:.4f}")

if r2_diff > 0.1:
    print("\n⚠️  ALERTA: Posible overfitting")
elif r2_diff < 0.02:
    print("\n✅ El modelo generaliza bien")
else:
    print("\n🔍 Overfitting leve, aceptable")

# Visualización de residuos
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Predicciones vs Real
axes[0].scatter(y_test, y_test_pred, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', linewidth=2, label='Predicción perfecta')
axes[0].set_xlabel('Valor Real')
axes[0].set_ylabel('Predicción')
axes[0].set_title('Predicciones vs Valores Reales')
axes[0].legend()

# 2. Residuos
residuos = y_test - y_test_pred
axes[1].scatter(y_test_pred, residuos, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicción')
axes[1].set_ylabel('Residuo')
axes[1].set_title('Residuos vs Predicciones')

# 3. Distribución de residuos
axes[2].hist(residuos, bins=20, edgecolor='black', alpha=0.7)
axes[2].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[2].set_xlabel('Residuo')
axes[2].set_ylabel('Frecuencia')
axes[2].set_title('Distribución de Residuos')

plt.tight_layout()
plt.show()

# Verificación de normalidad de residuos
stat, p_value = stats.shapiro(residuos)
print(f"\nTest de normalidad de residuos (Shapiro-Wilk):")
print(f"   Estadístico: {stat:.4f}")
print(f"   p-value: {p_value:.4f}")
print(f"   ¿Residuos normales? {'SÍ' if p_value > 0.05 else 'NO'} (α = 0.05)")

In [ ]:
# Celda 7: Regresión Logística (predecir loyalty_signups)

print("=" * 60)
print("REGRESIÓN LOGÍSTICA: Prediciendo loyalty_class")
print("=" * 60)

# Crear variable binaria
df['loyalty_class'] = (df['loyalty_signups'] > 10).astype(int)

print(f"\nDistribución de clases:")
print(f"   Clase 0 (≤10 suscripciones): {(df['loyalty_class'] == 0).sum()} días")
print(f"   Clase 1 (>10 suscripciones): {(df['loyalty_class'] == 1).sum()} días")
print(f"   Balance: {df['loyalty_class'].mean():.2%} positivos")

# Preparar datos
X_log = df[['transactions', 'avg_order_value', 'total_revenue', 'is_holiday']]
y_log = df['loyalty_class']

# Dividir
X_log_train, X_log_test, y_log_train, y_log_test = train_test_split(
    X_log, y_log, test_size=0.2, random_state=42, stratify=y_log
)

# Estandarizar
scaler_log = StandardScaler()
X_log_train_scaled = scaler_log.fit_transform(X_log_train)
X_log_test_scaled = scaler_log.transform(X_log_test)

# Entrenar modelo
log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(X_log_train_scaled, y_log_train)

# Predicciones
y_log_train_pred = log_model.predict(X_log_train_scaled)
y_log_test_pred = log_model.predict(X_log_test_scaled)
y_log_test_proba = log_model.predict_proba(X_log_test_scaled)[:, 1]

print("\nModelo entrenado exitosamente.")

# Coeficientes y odds ratios
coef_log_df = pd.DataFrame({
    'Variable': ['Intercepto'] + list(X_log.columns),
    'Coeficiente': [log_model.intercept_[0]] + list(log_model.coef_[0])
})
coef_log_df['Odds Ratio'] = np.exp(coef_log_df['Coeficiente'])

print("\nCoeficientes y Odds Ratios:")
print(coef_log_df.to_string(index=False))

print("\n" + "-" * 60)
print("INTERPRETACIÓN DE ODDS RATIOS:")
print("-" * 60)

for idx, row in coef_log_df.iterrows():
    if row['Variable'] != 'Intercepto':
        print(f"\n{row['Variable']}:")
        print(f"   Odds Ratio = {row['Odds Ratio']:.3f}")
        if row['Odds Ratio'] > 1:
            print(f"   → Aumenta la probabilidad en {((row['Odds Ratio'] - 1) * 100):.1f}%")
        else:
            print(f"   → Reduce la probabilidad en {((1 - row['Odds Ratio']) * 100):.1f}%")

In [ ]:
# Celda 8: Matriz de confusión

print("=" * 60)
print("MATRIZ DE CONFUSIÓN Y MÉTRICAS DE CLASIFICACIÓN")
print("=" * 60)

# Matriz de confusión
cm = confusion_matrix(y_log_test, y_log_test_pred)

print("\nMatriz de Confusión:")
print("                    Predicho")
print("                 Negativo  Positivo")
print(f"Real Negativo     {cm[0,0]:6d}    {cm[0,1]:6d}")
print(f"Real Positivo     {cm[1,0]:6d}    {cm[1,1]:6d}")

# Métricas
accuracy = accuracy_score(y_log_test, y_log_test_pred)
precision = precision_score(y_log_test, y_log_test_pred)
recall = recall_score(y_log_test, y_log_test_pred)
f1 = f1_score(y_log_test, y_log_test_pred)

print(f"\nMétricas de Clasificación:")
print(f"   Accuracy:  {accuracy:.4f} ({accuracy:.1%})")
print(f"   Precision: {precision:.4f} ({precision:.1%})")
print(f"   Recall:    {recall:.4f} ({recall:.1%})")
print(f"   F1-Score:  {f1:.4f} ({f1:.1%})")

print(f"\nInterpretación:")
print(f"   - Accuracy: {accuracy:.1%} de las predicciones fueron correctas")
print(f"   - Precision: De todos los que predije como positivos, {precision:.1%} realmente lo son")
print(f"   - Recall: Detecté el {recall:.1%} de todos los positivos reales")
print(f"   - F1: Media armónica entre precision y recall")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Matriz de confusión
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Negativo', 'Positivo'],
            yticklabels=['Negativo', 'Positivo'])
axes[0].set_xlabel('Predicho')
axes[0].set_ylabel('Real')
axes[0].set_title('Matriz de Confusión')

# 2. Curva ROC
fpr, tpr, thresholds = roc_curve(y_log_test, y_log_test_proba)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Clasificador aleatorio')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('Tasa de Falsos Positivos')
axes[1].set_ylabel('Tasa de Verdaderos Positivos')
axes[1].set_title('Curva ROC')
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

print(f"\nAUC-ROC: {roc_auc:.4f}")
print(f"Interpretación: {'Excelente' if roc_auc > 0.9 else 'Bueno' if roc_auc > 0.8 else 'Aceptable' if roc_auc > 0.7 else 'Malo'}")

In [ ]:
# Celda 9: Comparación de modelos

print("=" * 60)
print("COMPARACIÓN DE MODELOS")
print("=" * 60)

# Resumen de métricas
print("\n1. RESUMEN DE MÉTRICAS:")
print("\n   REGRESIÓN LINEAL (predecir total_revenue):")
print(f"   - R² Train: {metrics['R² (Train)']:.4f}")
print(f"   - R² Test:  {metrics['R² (Test)']:.4f}")
print(f"   - RMSE Test: ${metrics['RMSE (Test)']:.2f}")
print(f"   - MAE Test:  ${metrics['MAE (Test)']:.2f}")

print("\n   REGRESIÓN LOGÍSTICA (predecir loyalty_class):")
print(f"   - Accuracy:  {accuracy:.4f}")
print(f"   - Precision: {precision:.4f}")
print(f"   - Recall:    {recall:.4f}")
print(f"   - F1-Score:  {f1:.4f}")
print(f"   - AUC-ROC:   {roc_auc:.4f}")

# Interpretabilidad
print("\n2. INTERPRETABILIDAD:")
print("\n   Regresión Lineal:")
print("   ✅ Altamente interpretable")
print("   ✅ Cada coeficiente tieneUnits Units Units Units Units Units Units Inches UNits Units Units Units Units Units Units Units Units Inches Units Units UNits Units Units Units Units Units UNits Units Units Units Units Units Units UNits Units Units Units Units Units UNits Units Units UNits Units Units Units Units UNits Units Units Units Units UNits")

In [ ]:
# Celda 10: Post-Mortem: ¿Qué hizo mal?

print("=" * 60)
print("POST-MORTEM: ¿QUÉ HIZO MAL?")
print("=" * 60)

print("\n1. ERRORES COMETIDOS:")
print("   - ¿Se violaron supuestos de regresión?")
print("     → Verificar normalidad de residuos (Test Shapiro-Wilk)")
print("     → Verificar homocedasticidad (gráfico de residuos)")
print("     → Verificar independencia (autocorrelación)")

print("\n   - ¿Hay variables que deberían haberse excluido?")
print("     → total_revenue en regresión logística es problemático")
print("       (correlacionado con loyalty_signups)")

print("\n   - ¿Se manejaron correctamente los outliers?")
print("     → Días festivos con valores extremos afectan el modelo")
print("     → Deberíamos haber analizado si son errores o información real")

print("\n   - ¿Se verificó la ética del modelo?")
print("     → No se verificó fairness")
print("     → No se analizó si el modelo discrimina")

print("\n2. LECCIONES APRENDIDAS:")
print("   - SIEMPRE empezar con regresión lineal")
print("   - Los coeficientes cuentan la historia del negocio")
print("   - No usar variables que son consecuencia de la objetivo")
print("   - Verificar supuestos antes de confiar en el modelo")
print("   - La ética NO es opcional")

print("\n3. MEJORAS PROPUESTAS:")
print("   - Agregar variables: day_of_week codificado, weather codificado")
print("   - Usar regularización (Ridge/Lasso) si hay multicolinealidad")
print("   - Implementar validación cruzada temporal")
print("   - Analizar residuos con más profundidad")

print("\n4. ÉTICA:")
print("   - ¿Hay sesgo en las variables de entrada?")
print("     → Analizar si is_holiday está distribuido equitativamente")
print("   - ¿El modelo podría discriminar?")
print("     → Verificar fairness por grupo")
print("   - ¿Se puede explicar cada predicción?")
print("     → SÍ: ambas regresiones son explicables")

print("\n5. VEREDICTO:")
print("   La regresión lineal CUMPLE como prueba de fuego.")
print(f"   R² = {metrics['R² (Test)']:.4f} es {'excelente' if metrics['R² (Test)'] > 0.8 else 'bueno' if metrics['R² (Test)'] > 0.6 else 'mejorable'}.")
print("   Un modelo complejo debe superar significativamente estos resultados.")

print("\n" + "=" * 60)
print("FIN DEL NOTEBOOK")
print("=" * 60)